# Exp38R3 — Complete 120-condition fixed-trust control for PRR and PRR-Stat

This notebook assembles the exact 120-condition **common-trust control** used in the appendix. It does not select the final paper trust coefficient.

## Inputs

- exact Exp33 query/channel manifests;
- frozen direct forecasts for five backbones;
- PRR-Stat test retrieval caches from Exp36R;
- PRR validation/test retrieval caches from Exp35B.

## Controlled downstream comparison

Both retrieval methods are combined with each frozen forecaster using the same fixed trust

\[
\hat y=d+0.1(r-d).
\]

This yields a controlled 120-condition comparison of Direct, PRR-Stat, and PRR with no model retraining or forecaster inference.

The **main paper protocol** is evaluated later in `06_condition_trust_calibration.ipynb`, where one bounded trust coefficient \(\beta\) per dataset × horizon × backbone is selected on validation only and then frozen on test.

In [ ]:

# Cell 1 — setup

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("PRR_ROOT", str(Path.cwd().resolve().parents[1] if Path.cwd().name == "prr_long_horizon" else Path.cwd().resolve())))

MANIFEST_ROOT = ROOT / "results" / "exp33" / "final_eval_manifest"
PATCHTST_DIRECT_ROOT = ROOT / "results" / "exp34" / "patchtst_direct_cache"
OTHER_DIRECT_ROOT = ROOT / "results" / "exp37D" / "direct_cache"

PRR_STAT_CACHE_ROOT = ROOT / "results" / "exp36R" / "prr_stat_retrieval_cache"
PRR_STAT_RESULT_CSV = ROOT / "results" / "exp36R" / "patchtst24_prr_stat_vs_prr.csv"

PRR_PART_ROOT = ROOT / "results" / "exp35B" / "channel_parts"
PRR_RESULT_CSV = ROOT / "results" / "exp35B" / "patchtst_24_prr_results.csv"

OUT = ROOT / "results" / "exp38R"
OUT.mkdir(parents=True, exist_ok=True)

DATASETS = ("ETTh1","Weather","Electricity","Traffic","Exchange","Solar")
HORIZONS = (96,192,336,720)
BACKBONES = ("PatchTST","iTransformer","TimeMixer","SegMoE","DLinear")
SPLITS = ("val","test")

FIXED_BETA = 0.1
EPS = 1e-12

print("Output:", OUT)
print("PRR-Stat test cache:", PRR_STAT_CACHE_ROOT.exists())
print("PRR channel parts:", PRR_PART_ROOT.exists())

assert MANIFEST_ROOT.exists()
assert PRR_STAT_CACHE_ROOT.exists()
assert PRR_PART_ROOT.exists()


In [ ]:

# Cell 2 — Exp33 manifest loader

def load_manifest(dataset,horizon,split):
    p = MANIFEST_ROOT / f"{dataset}_H{horizon}_{split}.npz"
    assert p.exists(), p

    with np.load(p,allow_pickle=False) as z:
        a=np.asarray(z["anchor"],np.int64)
        c=np.asarray(z["channel"],np.int64)

    assert a.shape==c.shape
    return a,c

manifest_rows=[]

for d in DATASETS:
    for h in HORIZONS:
        for sp in SPLITS:
            a,c=load_manifest(d,h,sp)
            manifest_rows.append({
                "dataset":d,
                "horizon":h,
                "split":sp,
                "pairs":len(a),
                "channels":len(np.unique(c)),
            })

manifest_df=pd.DataFrame(manifest_rows)
display(manifest_df)

assert len(manifest_df)==48
print("[PASS] Exp33 manifests 48/48")


In [ ]:

# Cell 3 — exact direct-cache loader

def direct_slug(backbone):
    return backbone.lower().replace("-","").replace("_","")

def direct_path_candidates(backbone,dataset,horizon,split):
    root=(
        PATCHTST_DIRECT_ROOT
        if backbone=="PatchTST"
        else OTHER_DIRECT_ROOT
    )

    slug=direct_slug(backbone)

    exact=[
        root/f"{dataset}_H{horizon}_{split}_{slug}_direct.npz",
        root/f"{dataset}_H{horizon}_{split}_{backbone}_direct.npz",
        root/f"{dataset}_H{horizon}_{split}_direct.npz",
    ]

    out=[p for p in exact if p.exists()]

    # fallback only if exact naming differs
    if not out and root.exists():
        for p in root.rglob("*.npz"):
            s=p.name.lower()
            if (
                dataset.lower() in s
                and f"h{horizon}" in s
                and split in s
            ):
                out.append(p)

    return out

def load_direct(backbone,dataset,horizon,split):
    ma,mc=load_manifest(dataset,horizon,split)

    for p in direct_path_candidates(backbone,dataset,horizon,split):
        try:
            with np.load(p,allow_pickle=False) as z:
                if not {"anchor","channel","direct","truth"}.issubset(z.files):
                    continue

                a=np.asarray(z["anchor"],np.int64)
                c=np.asarray(z["channel"],np.int64)
                d=np.asarray(z["direct"],np.float32)
                y=np.asarray(z["truth"],np.float32)

            if not (
                np.array_equal(a,ma)
                and np.array_equal(c,mc)
                and d.shape==(len(ma),horizon)
                and y.shape==(len(ma),horizon)
                and np.isfinite(d).all()
                and np.isfinite(y).all()
            ):
                continue

            return {
                "anchor":a,
                "channel":c,
                "direct":d,
                "truth":y,
                "path":str(p),
            }

        except Exception:
            continue

    raise FileNotFoundError(
        f"Exact direct cache not found: "
        f"{backbone} {dataset} H={horizon} {split}"
    )

audit=[]

for b in BACKBONES:
    for d in DATASETS:
        for h in HORIZONS:
            ok=True
            for sp in SPLITS:
                try:
                    load_direct(b,d,h,sp)
                except Exception:
                    ok=False

            audit.append({
                "backbone":b,
                "dataset":d,
                "horizon":h,
                "ok":ok,
            })

direct_audit=pd.DataFrame(audit)

display(
    direct_audit.groupby("backbone",as_index=False)
    .agg(total=("ok","size"),ok=("ok","sum"))
)

assert int(direct_audit.ok.sum())==120
print("[PASS] direct cache 120/120")


In [ ]:

# Cell 4 — PRR-Stat test retrieval loader

def load_prr_stat_test(dataset,horizon):
    p=PRR_STAT_CACHE_ROOT/f"{dataset}_H{horizon}_test.npz"
    assert p.exists(),p

    ma,mc=load_manifest(dataset,horizon,"test")

    with np.load(p,allow_pickle=False) as z:
        a=np.asarray(z["anchor"],np.int64)
        c=np.asarray(z["channel"],np.int64)
        r=np.asarray(z["retrieval"],np.float32)

    assert np.array_equal(a,ma)
    assert np.array_equal(c,mc)
    assert r.shape==(len(ma),horizon)
    assert np.isfinite(r).all()

    return {
        "anchor":a,
        "channel":c,
        "retrieval":r,
        "path":str(p),
    }

prr_stat_audit=[]

for d in DATASETS:
    for h in HORIZONS:
        x=load_prr_stat_test(d,h)
        prr_stat_audit.append({
            "dataset":d,
            "horizon":h,
            "pairs":len(x["anchor"]),
            "path":x["path"],
        })

prr_stat_audit=pd.DataFrame(prr_stat_audit)
display(prr_stat_audit)

assert len(prr_stat_audit)==24
print("[PASS] PRR-Stat test retrieval 24/24")


In [ ]:

# Cell 5 — PRR per-channel cache assembler

def prr_channel_file(dataset,horizon,split,ch):
    return (
        PRR_PART_ROOT
        / dataset
        / f"H{horizon}"
        / split
        / f"ch{int(ch):04d}.npz"
    )

def load_prr(dataset,horizon,split):
    ma,mc=load_manifest(dataset,horizon,split)

    out=np.full(
        (len(ma),horizon),
        np.nan,
        dtype=np.float32,
    )

    analog=np.full(
        len(ma),
        np.nan,
        dtype=np.float32,
    )

    union_size=np.full(
        len(ma),
        np.nan,
        dtype=np.float32,
    )

    learned_only=np.full(
        len(ma),
        np.nan,
        dtype=np.float32,
    )

    used=[]

    for ch in np.unique(mc):
        rows=np.flatnonzero(mc==ch)
        expected_anchor=ma[rows]

        p=prr_channel_file(
            dataset,
            horizon,
            split,
            int(ch),
        )

        assert p.exists(),p

        with np.load(p,allow_pickle=False) as z:
            assert "anchor" in z.files
            assert "retrieval" in z.files

            a=np.asarray(z["anchor"],np.int64)
            r=np.asarray(z["retrieval"],np.float32)

            assert r.shape==(len(a),horizon),(p,r.shape)

            # Normal case: channel part follows Exp33 row order within channel.
            if np.array_equal(a,expected_anchor):
                order=np.arange(len(a),dtype=np.int64)
            else:
                # Robust fallback: match unique anchors explicitly.
                # Exp33 samples anchors without replacement within channel in
                # the standard protocol, but this also makes mismatch visible.
                pos={}
                for j,v in enumerate(a):
                    pos.setdefault(int(v),[]).append(j)

                order=[]
                for v in expected_anchor:
                    bucket=pos.get(int(v),[])
                    if not bucket:
                        raise RuntimeError(
                            f"Learned cache anchor mismatch: "
                            f"{dataset} H={horizon} {split} ch={ch}, "
                            f"missing anchor {int(v)}"
                        )
                    order.append(bucket.pop(0))

                order=np.asarray(order,np.int64)

            out[rows]=r[order]

            if "analog_mse" in z.files:
                v=np.asarray(z["analog_mse"],np.float32)
                analog[rows]=v[order]

            if "union_size" in z.files:
                v=np.asarray(z["union_size"],np.float32)
                union_size[rows]=v[order]

            if "learned_only" in z.files:
                v=np.asarray(z["learned_only"],np.float32)
                learned_only[rows]=v[order]

        used.append(str(p))

    assert np.isfinite(out).all()
    assert out.shape==(len(ma),horizon)

    return {
        "anchor":ma,
        "channel":mc,
        "retrieval":out,
        "analog_mse":analog,
        "union_size":union_size,
        "learned_only":learned_only,
        "files":used,
    }

prr_cache={}
learned_audit=[]

for d in DATASETS:
    for h in HORIZONS:
        for sp in SPLITS:
            x=load_prr(d,h,sp)
            prr_cache[(d,h,sp)]=x

            learned_audit.append({
                "dataset":d,
                "horizon":h,
                "split":sp,
                "pairs":len(x["anchor"]),
                "channels":len(np.unique(x["channel"])),
                "source_files":len(x["files"]),
                "mean_union_size":
                    float(np.nanmean(x["union_size"]))
                    if np.isfinite(x["union_size"]).any()
                    else np.nan,
                "mean_learned_only":
                    float(np.nanmean(x["learned_only"]))
                    if np.isfinite(x["learned_only"]).any()
                    else np.nan,
            })

learned_audit=pd.DataFrame(learned_audit)

display(
    learned_audit.groupby("split",as_index=False)
    .agg(
        conditions=("dataset","size"),
        total_source_files=("source_files","sum"),
        mean_union_size=("mean_union_size","mean"),
        mean_learned_only=("mean_learned_only","mean"),
    )
)

assert len(learned_audit)==48
print("[PASS] PRR retrieval val/test = 48/48")


In [ ]:

# Cell 6 — metric / integration helpers

def mse(a,b):
    a=np.asarray(a,np.float64)
    b=np.asarray(b,np.float64)
    return float(np.mean((a-b)**2))

def mae(a,b):
    a=np.asarray(a,np.float64)
    b=np.asarray(b,np.float64)
    return float(np.mean(np.abs(a-b)))

def combine(direct,retrieval,beta):
    d=np.asarray(direct,np.float32)
    r=np.asarray(retrieval,np.float32)
    return d+float(beta)*(r-d)

def global_beta(direct,retrieval,truth):
    d=np.asarray(direct,np.float64)
    r=np.asarray(retrieval,np.float64)
    y=np.asarray(truth,np.float64)

    delta=r-d
    denom=np.sum(delta*delta)

    if not np.isfinite(denom) or denom<=EPS:
        return 0.0

    numer=-np.sum((d-y)*delta)

    return float(
        np.clip(
            numer/denom,
            0.0,
            1.0,
        )
    )

def wtl(series,tol=1e-10):
    x=pd.Series(series).dropna().to_numpy(float)

    return {
        "W":int((x>tol).sum()),
        "T":int((np.abs(x)<=tol).sum()),
        "L":int((x<-tol).sum()),
    }


In [ ]:

# Cell 7 — retrieval quality, 24 backbone-independent conditions
#
# PRIMARY retrieval metric:
#   Exp36R already saved the exact matched-protocol
#   `retrieval_gain_prr_vs_prr_stat_pct`.
#
# The merged Exp36R CSV does NOT retain the two raw AnalogFutureMSE
# columns separately. Therefore we use the saved gain directly as the
# source of truth for the primary retrieval comparison.
#
# SECONDARY diagnostic:
#   RetrievalForecastMSE after averaging Top-K candidate futures.
#   This can be recomputed exactly from the persisted retrieval caches.

assert PRR_STAT_RESULT_CSV.exists(), PRR_STAT_RESULT_CSV

ref = pd.read_csv(PRR_STAT_RESULT_CSV)
ref["horizon"] = ref["horizon"].astype(int)

required = {
    "dataset",
    "horizon",
    "retrieval_gain_prr_vs_prr_stat_pct",
}
missing = required - set(ref.columns)
assert not missing, f"Missing Exp36R columns: {sorted(missing)}"

rows = []

for d in DATASETS:
    for h in HORIZONS:
        rr = ref[
            (ref.dataset == d)
            & (ref.horizon == h)
        ]
        assert len(rr) == 1, (d, h, len(rr))
        rr = rr.iloc[0]

        # ------------------------------------------------------------
        # (A) PRIMARY: AnalogFutureMSE gain saved by Exp36R
        # ------------------------------------------------------------
        analog_gain = float(
            rr["retrieval_gain_prr_vs_prr_stat_pct"]
        )

        # Learned analog MSE is still available in Exp35B per-channel caches.
        prr_obj = prr_cache[(d,h,"test")]
        prr_analog_vec = np.asarray(
            prr_obj["analog_mse"],
            dtype=np.float64,
        )
        assert np.isfinite(prr_analog_vec).all(), (d,h)
        prr_analog_mse = float(
            prr_analog_vec.mean()
        )

        # Reconstruct the corresponding PRR-Stat AnalogFutureMSE only for
        # descriptive display. This follows exactly from the stored gain:
        #
        # gain = 100 * (PRR-Stat - PRR) / PRR-Stat
        #
        # => PRR-Stat = PRR / (1 - gain/100)
        denom = 1.0 - analog_gain/100.0
        prr_stat_analog_mse_reconstructed = (
            prr_analog_mse / denom
            if abs(denom) > 1e-12
            else np.nan
        )

        # ------------------------------------------------------------
        # (B) SECONDARY: retrieval forecast aggregation diagnostic
        # ------------------------------------------------------------
        truth = load_direct(
            "PatchTST",
            d,h,"test",
        )["truth"]

        prr_stat_retrieval = load_prr_stat_test(
            d,h
        )["retrieval"]

        prr_retrieval = prr_obj[
            "retrieval"
        ]

        prr_stat_forecast_mse = mse(
            prr_stat_retrieval,
            truth,
        )

        prr_forecast_mse = mse(
            prr_retrieval,
            truth,
        )

        forecast_gain = (
            100.0
            * (
                prr_stat_forecast_mse
                - prr_forecast_mse
            )
            / prr_stat_forecast_mse
        )

        rows.append({
            "dataset":d,
            "horizon":h,

            # Primary retrieval-quality result
            "prr_analog_future_mse":
                prr_analog_mse,
            "prr_stat_analog_future_mse_reconstructed":
                prr_stat_analog_mse_reconstructed,
            "prr_vs_prr_stat_analog_gain_pct":
                analog_gain,

            # Secondary aggregation diagnostic
            "prr_stat_retrieval_forecast_mse":
                prr_stat_forecast_mse,
            "prr_retrieval_forecast_mse":
                prr_forecast_mse,
            "prr_vs_prr_stat_retrieval_forecast_gain_pct":
                forecast_gain,

            "mean_union_size":
                float(
                    np.nanmean(
                        prr_obj["union_size"]
                    )
                ),

            "mean_learned_only_candidates":
                float(
                    np.nanmean(
                        prr_obj["learned_only"]
                    )
                ),
        })

retrieval24 = pd.DataFrame(rows)

display(retrieval24)

# Primary paper-facing retrieval result
g = retrieval24[
    "prr_vs_prr_stat_analog_gain_pct"
]

print("\n(A) AnalogFutureMSE — PRIMARY RETRIEVAL QUALITY")
print("W/T/L:", wtl(g))
print("Mean gain %  :", float(g.mean()))
print("Median gain %:", float(g.median()))

# Secondary aggregation diagnostic
gf = retrieval24[
    "prr_vs_prr_stat_retrieval_forecast_gain_pct"
]

print("\n(B) RetrievalForecastMSE — AGGREGATION DIAGNOSTIC")
print("W/T/L:", wtl(gf))
print("Mean gain %  :", float(gf.mean()))
print("Median gain %:", float(gf.median()))

print(
    "\nMean learned-only candidates:",
    float(
        retrieval24[
            "mean_learned_only_candidates"
        ].mean()
    ),
)

retrieval24.to_csv(
    OUT / "retrieval_quality_24conditions_final.csv",
    index=False,
)


In [ ]:

# Cell 8 — final fair fixed-beta 120-condition comparison

rows=[]

for b in BACKBONES:
    for d in DATASETS:
        for h in HORIZONS:
            dt=load_direct(b,d,h,"test")

            direct=dt["direct"]
            truth=dt["truth"]

            prr_stat=load_prr_stat_test(d,h)["retrieval"]
            prr=prr_cache[(d,h,"test")]["retrieval"]

            op=combine(direct,prr_stat,FIXED_BETA)
            lp=combine(direct,prr,FIXED_BETA)

            dm=mse(direct,truth)
            om=mse(op,truth)
            lm=mse(lp,truth)

            rows.append({
                "backbone":b,
                "dataset":d,
                "horizon":h,

                "direct_mse":dm,
                "direct_mae":mae(direct,truth),

                "prr_stat_fixed01_mse":om,
                "prr_stat_fixed01_mae":mae(op,truth),
                "prr_stat_fixed01_gain_vs_direct_pct":
                    100.0*(dm-om)/dm,

                "prr_fixed01_mse":lm,
                "prr_fixed01_mae":mae(lp,truth),
                "prr_fixed01_gain_vs_direct_pct":
                    100.0*(dm-lm)/dm,

                "prr_vs_prr_stat_fixed01_gain_pct":
                    100.0*(om-lm)/om,
            })

fixed120=pd.DataFrame(rows)

assert len(fixed120)==120
assert fixed120.notna().all().all()

display(fixed120.head(20))

fixed120.to_csv(
    OUT/"fixed_beta01_120_conditions.csv",
    index=False,
)

print("[PASS] fair fixed-beta comparison = 120/120")


In [ ]:

# Cell 9 — PRR validation-only GlobalAnalytic, all 120 conditions
#
# This is secondary analysis only.
# It is NOT used as the primary PRR-vs-PRR-Stat comparison because
# PRR-Stat validation retrieval was not persisted in Exp36R.

rows=[]

for b in BACKBONES:
    for d in DATASETS:
        for h in HORIZONS:
            dv=load_direct(b,d,h,"val")
            dt=load_direct(b,d,h,"test")

            rv=prr_cache[(d,h,"val")]["retrieval"]
            rt=prr_cache[(d,h,"test")]["retrieval"]

            beta=global_beta(
                dv["direct"],
                rv,
                dv["truth"],
            )

            pred=combine(
                dt["direct"],
                rt,
                beta,
            )

            dm=mse(
                dt["direct"],
                dt["truth"],
            )

            gm=mse(
                pred,
                dt["truth"],
            )

            rows.append({
                "backbone":b,
                "dataset":d,
                "horizon":h,
                "prr_global_beta":beta,
                "direct_mse":dm,
                "prr_global_mse":gm,
                "prr_global_mae":
                    mae(pred,dt["truth"]),
                "prr_global_gain_vs_direct_pct":
                    100.0*(dm-gm)/dm,
            })

prr_global120=pd.DataFrame(rows)

assert len(prr_global120)==120
assert prr_global120.notna().all().all()

display(prr_global120.head(20))

prr_global120.to_csv(
    OUT/"prr_global_analytic_120_conditions.csv",
    index=False,
)

print("[PASS] PRR GlobalAnalytic = 120/120")


In [ ]:

# Cell 10 — exact sanity checks against validated Exp36R
#
# What can be independently reproduced from persisted arrays:
#   1) PRR-Stat fixed-beta downstream MSE
#   2) PRR fixed-beta downstream MSE
#
# The Exp36R merged CSV stores the AnalogFutureMSE gain but not both
# raw AnalogFutureMSE values, so we do NOT invent a raw-value assertion.

assert PRR_STAT_RESULT_CSV.exists(), PRR_STAT_RESULT_CSV

ref = pd.read_csv(PRR_STAT_RESULT_CSV)
ref["horizon"] = ref["horizon"].astype(int)

# ------------------------------------------------------------
# 1. Fixed-beta downstream MSE reproduction
# ------------------------------------------------------------

cur = fixed120[
    fixed120.backbone == "PatchTST"
].copy()

needed = {
    "dataset",
    "horizon",
    "prr_stat_fixed01_mse",
    "prr_fixed01_mse",
    "retrieval_gain_prr_vs_prr_stat_pct",
}
missing = needed - set(ref.columns)
assert not missing, f"Missing Exp36R columns: {sorted(missing)}"

chk = cur.merge(
    ref[
        [
            "dataset",
            "horizon",
            "prr_stat_fixed01_mse",
            "prr_fixed01_mse",
        ]
    ],
    on=["dataset","horizon"],
    suffixes=("_new","_ref"),
)

assert len(chk) == 24

chk["prr_stat_absdiff"] = np.abs(
    chk.prr_stat_fixed01_mse_new
    - chk.prr_stat_fixed01_mse_ref
)

chk["prr_absdiff"] = np.abs(
    chk.prr_fixed01_mse_new
    - chk.prr_fixed01_mse_ref
)

max_o = float(
    chk.prr_stat_absdiff.max()
)

max_l = float(
    chk.prr_absdiff.max()
)

print(
    "Max PRR-Stat fixed-MSE diff:",
    max_o,
)

print(
    "Max PRR fixed-MSE diff :",
    max_l,
)

assert max_o < 5e-6
assert max_l < 5e-6

# ------------------------------------------------------------
# 2. Retrieval-gain identity check
# ------------------------------------------------------------
#
# Cell 7 intentionally imports Exp36R's exact matched-protocol
# AnalogFutureMSE gain because the merged CSV retained the gain
# but not both raw analog MSE columns.
#
# Here we verify the 24 condition identities and values exactly.

rg = retrieval24.merge(
    ref[
        [
            "dataset",
            "horizon",
            "retrieval_gain_prr_vs_prr_stat_pct",
        ]
    ],
    on=["dataset","horizon"],
    how="inner",
)

assert len(rg) == 24

rg["gain_absdiff"] = np.abs(
    rg.prr_vs_prr_stat_analog_gain_pct
    - rg.retrieval_gain_prr_vs_prr_stat_pct
)

max_rg = float(
    rg.gain_absdiff.max()
)

print(
    "Max Analog retrieval-gain diff:",
    max_rg,
)

assert max_rg < 1e-12

print(
    "\n[PASS] Exp38R3 reproduces the validated "
    "PatchTST fixed-beta results exactly and carries forward "
    "the exact Exp36R AnalogFutureMSE retrieval gain."
)


In [ ]:

# Cell 11 — overall fixed-beta summary

gain=fixed120[
    "prr_vs_prr_stat_fixed01_gain_pct"
]

o_direct=fixed120[
    "prr_stat_fixed01_gain_vs_direct_pct"
]

l_direct=fixed120[
    "prr_fixed01_gain_vs_direct_pct"
]

overall=pd.DataFrame([
    {
        "comparison":"PRR vs PRR-Stat",
        "conditions":len(gain),
        **wtl(gain),
        "mean_gain_pct":gain.mean(),
        "median_gain_pct":gain.median(),
    },
    {
        "comparison":"PRR-Stat vs Direct",
        "conditions":len(o_direct),
        **wtl(o_direct),
        "mean_gain_pct":o_direct.mean(),
        "median_gain_pct":o_direct.median(),
    },
    {
        "comparison":"PRR vs Direct",
        "conditions":len(l_direct),
        **wtl(l_direct),
        "mean_gain_pct":l_direct.mean(),
        "median_gain_pct":l_direct.median(),
    },
])

display(overall)
overall.to_csv(
    OUT/"overall_fixed_beta_summary.csv",
    index=False,
)


In [ ]:

# Cell 12 — dataset / backbone / horizon summaries

def group_summary(df,col):
    rows=[]

    for key,g in df.groupby(col):
        comp=g[
            "prr_vs_prr_stat_fixed01_gain_pct"
        ]

        c=wtl(comp)

        rows.append({
            col:key,
            "conditions":len(g),
            "W":c["W"],
            "T":c["T"],
            "L":c["L"],
            "prr_vs_prr_stat_mean_gain_pct":
                comp.mean(),
            "prr_vs_prr_stat_median_gain_pct":
                comp.median(),
            "prr_stat_gain_vs_direct_mean_pct":
                g.prr_stat_fixed01_gain_vs_direct_pct.mean(),
            "prr_gain_vs_direct_mean_pct":
                g.prr_fixed01_gain_vs_direct_pct.mean(),
        })

    return pd.DataFrame(rows)

dataset_summary=group_summary(
    fixed120,
    "dataset",
)

backbone_summary=group_summary(
    fixed120,
    "backbone",
)

horizon_summary=group_summary(
    fixed120,
    "horizon",
)

print("DATASET SUMMARY")
display(dataset_summary)

print("BACKBONE SUMMARY")
display(backbone_summary)

print("HORIZON SUMMARY")
display(horizon_summary)

dataset_summary.to_csv(
    OUT/"dataset_summary_fixed_beta01.csv",
    index=False,
)

backbone_summary.to_csv(
    OUT/"backbone_summary_fixed_beta01.csv",
    index=False,
)

horizon_summary.to_csv(
    OUT/"horizon_summary_fixed_beta01.csv",
    index=False,
)


In [ ]:

# Cell 13 — combined paper-facing 120-condition table

paper=fixed120.merge(
    prr_global120[
        [
            "backbone",
            "dataset",
            "horizon",
            "prr_global_beta",
            "prr_global_mse",
            "prr_global_gain_vs_direct_pct",
        ]
    ],
    on=["backbone","dataset","horizon"],
    how="left",
)

assert len(paper)==120
assert paper[
    [
        "direct_mse",
        "prr_stat_fixed01_mse",
        "prr_fixed01_mse",
    ]
].notna().all().all()

paper.to_csv(
    OUT/"paper_table_120_conditions.csv",
    index=False,
)

display(paper)

print("[PASS] paper-facing 120-condition table saved.")


In [ ]:

# Cell 14 — final scientific readout

retr_gain = retrieval24[
    "prr_vs_prr_stat_analog_gain_pct"
]

forecast_gain = fixed120[
    "prr_vs_prr_stat_fixed01_gain_pct"
]

summary={
    "experiment":"Exp38R3",

    "direct_conditions":120,
    "prr_stat_fixed_conditions":120,
    "prr_fixed_conditions":120,
    "prr_global_conditions":120,

    "retrieval_conditions":24,
    "retrieval_wtl":wtl(retr_gain),
    "retrieval_mean_gain_pct":
        float(retr_gain.mean()),
    "retrieval_median_gain_pct":
        float(retr_gain.median()),

    "retrieval_metric":
        "AnalogFutureMSE",

    "retrieval_forecast_diagnostic_wtl":
        wtl(
            retrieval24[
                "prr_vs_prr_stat_retrieval_forecast_gain_pct"
            ]
        ),

    "retrieval_forecast_diagnostic_mean_gain_pct":
        float(
            retrieval24[
                "prr_vs_prr_stat_retrieval_forecast_gain_pct"
            ].mean()
        ),

    "forecast_fixed_prr_vs_prr_stat_wtl":
        wtl(forecast_gain),
    "forecast_fixed_prr_vs_prr_stat_mean_gain_pct":
        float(forecast_gain.mean()),
    "forecast_fixed_prr_vs_prr_stat_median_gain_pct":
        float(forecast_gain.median()),

    "prr_stat_fixed_vs_direct_wtl":
        wtl(
            fixed120.prr_stat_fixed01_gain_vs_direct_pct
        ),

    "prr_fixed_vs_direct_wtl":
        wtl(
            fixed120.prr_fixed01_gain_vs_direct_pct
        ),

    "mean_learned_only_candidates":
        float(
            retrieval24.mean_learned_only_candidates.mean()
        ),

    "training_performed":False,
    "forecaster_inference_performed":False,
}

(OUT/"exp38R_summary.json").write_text(
    json.dumps(summary,indent=2),
    encoding="utf-8",
)

print(json.dumps(summary,indent=2))

print("\n"+"="*110)
print("EXP38R3 FINAL STATUS")
print("="*110)
print("Direct                         : 120/120")
print("PRR-Stat fixed beta=0.1  : 120/120")
print("PRR fixed beta=0.1   : 120/120")
print("PRR GlobalAnalytic    : 120/120")
print("Retrieval quality comparison  : 24/24")
print("\nNo model training or forecaster inference was performed.")
print("Saved under:",OUT)
